In [1]:
import pandas as pd
import numpy as np
import os
from scipy.stats import pearsonr
import seaborn as sns
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.tree import DecisionTreeRegressor

# 讀入資料

In [ ]:
# 1. 載入資料集（請修改成你的資料檔案路徑）
# "C:\Users\user\Desktop\0716\mix.csv"
# path = "C:/Users/user/Desktop/0712_tw240_ss12/mix_0712_xface_xratio(spo2).csv"

path = "C:/Users/user/Desktop/0716/mix(hr).csv"

df = pd.read_csv(path)  # 假設CSV裡已經有特徵與目標變數

In [ ]:
# 計算資料筆數與欄位數
df.shape

# 資料集敘述統計 
# df.describe().T

# 資料前處理

In [ ]:
# 刪除異常值(平均+-2倍標準差)

def mod_outliers_replace_with_median(df, factor=1.5, exclude_columns=None):

    # 分出要處理與不處理的欄位
    if exclude_columns is None:
        exclude_columns = []

    # 只選數值欄位
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col not in exclude_columns]
    
    # 建立一份副本
    df_pre = df.copy()
    
    # 計算 IQR, 下界與上界
    # Q1 = df[numeric_cols].quantile(0.25)
    # Q3 = df[numeric_cols].quantile(0.75)
    # IQR = Q3 - Q1
    # lower_bound = Q1 - factor * IQR
    # upper_bound = Q3 + factor * IQR

    # 替換離群值為平均值
    for col in numeric_cols:
        median_val = df[col].median()
        
        # 偵測異常值條件
        # outlier_condition = (df[col] < lower_bound[col]) | (df[col] > upper_bound[col])
        null_condition = df[col].isna()
        # zero_condition = (df[col] == 0)
        
        # 統計異常與空值
        # outlier_count = outlier_condition.sum()
        null_count = null_condition.sum()
        # zero_count = zero_condition.sum()
        
        # 用中位數取代
        # df_replaced.loc[outlier_condition, col] = median_val
        df_pre.loc[null_condition, col] = median_val
        # df_pre.loc[zero_condition, col] = median_val

        # print(f"{col}: {outlier_count} outliers + {null_count} NaNs replaced with median ({median_val:.2f})")
        # print(f"{col}: {outlier_count} outliers + {null_count} NaNs + {zero_count} zeros replaced with median ({median_val:.2f})")
        # print(f"{col}:  {null_count} NaNs + {zero_count} zeros replaced with median ({median_val:.2f})")
        print(f"{col}:  {null_count} NaNs replaced with median ({median_val:.2f})")


    print(f"處理完成！異常值與空值皆以中位數取代 (factor={factor})")
    print(f"處理欄位數量：{len(numeric_cols)}")
    
    return df_pre

exclude_columns = ['case','segment','hr']  
df_pre = mod_outliers_replace_with_median(df, factor=2.0, exclude_columns =exclude_columns)
print(df_pre.shape)

In [ ]:
# 2. 特徵與標籤分離（X: 特徵；y: 標籤）

# 假設資料包含以下欄位：'編號', '時窗編號', '特徵1', '特徵2', ..., '目標值'
# 確保 '編號' 和 '時窗編號' 不作為特徵值
identifier_columns = ['case', 'segment']
feature_columns = [col for col in df.columns if col not in identifier_columns + ['hr']]

X = df_pre[feature_columns]
y = df_pre['hr']
identifiers = df[identifier_columns]

# 3. 切分訓練集與測試集（80% 訓練 / 20% 測試）
X_train, X_test, y_train, y_test, identifiers_train, identifiers_test = train_test_split(X, y, identifiers, test_size=0.2, random_state=42)

print("X_train.shape =", X_train.shape)
print("X_test.shape =", X_test.shape)


### 描述性統計

In [ ]:
import pandas as pd

# === 建立 person_id（如 '01-1' → '01'）
df_pre['person_id'] = df_pre['case'].astype(str).apply(lambda x: x.split('-')[0])

# === 建立分類函數
def classify_pid(pid):
    digits = ''.join(filter(str.isdigit, str(pid)))
    if 2 <= len(digits) < 5:
        return '健康人'
    else:
        return '病人'

df_pre['身分類別'] = df_pre['person_id'].apply(classify_pid)

# === 計算人數（唯一 person_id 數）
people_counts = df_pre[['person_id', '身分類別']].drop_duplicates()['身分類別'].value_counts()

# === 計算 case 數（唯一 case 數）
case_counts = df_pre[['case', '身分類別']].drop_duplicates()['身分類別'].value_counts()

# === 計算 seg 數（每列為一筆）
seg_counts = df_pre['身分類別'].value_counts()

# === 合併為統計表（加上總數列）
summary_df = pd.DataFrame({
    '人數（person_id）': people_counts,
    '量測次數（case）': case_counts,
    '資料筆數（seg）': seg_counts
}).fillna(0).astype(int)

summary_df.loc['總計'] = summary_df.sum()

# === 顯示結果
print("📊 健康人與病人的統計表（含總計）：")
print(summary_df)

# ✅（可選）儲存為 CSV
# summary_df.to_csv('健康與病人統計表.csv', encoding='utf-8-sig')


In [ ]:
# 計算基本描述統計
feature_stats = df_pre.describe().T  # 轉置後行是欄位名稱

# 加入範圍欄位（最大值 - 最小值）
feature_stats['range'] = feature_stats['max'] - feature_stats['min']

# 只取我們關心的欄位
feature_summary = feature_stats[['mean', 'std', 'min', 'max', 'range']].round(2)

# 輸出為 CSV
output_path = "特徵統計分析結果.csv"
feature_summary.to_csv(output_path, encoding='utf-8-sig')

print(f"✅ 已輸出統計資訊到檔案：{output_path}")

In [ ]:
# case轉字串，為了分析健康跟病人個數

# 假設你有新的完整資料集 new_df（包含 case 與 hr 及特徵欄位）
new_df = df_pre.copy()  # 你可替換成你想用的新資料集

# 特徵欄位與標籤分離（不含 case 與 segment）
id_cols = ['case', 'segment']
feature_cols = [col for col in new_df.columns if col not in id_cols + ['hr']]

X_all = new_df[feature_cols]
y_all = new_df['hr']
identifiers_all = new_df[id_cols]

from sklearn.model_selection import train_test_split

# 分割訓練測試集
X_train_new, X_test_new, y_train_new, y_test_new, id_train_new, id_test_new = train_test_split(
    X_all, y_all, identifiers_all, test_size=0.2, random_state=42
)

# 取出 case 欄位（轉字串）
case_train_new = id_train_new['case'].astype(str)
case_test_new = id_test_new['case'].astype(str)

# 建立標籤函數（健康人/病人判斷）
def classify_patient(pid):
    pid_digits = ''.join(filter(str.isdigit, pid))
    if 2 <= len(pid_digits) < 5:
        return '健康人'
    else:
        return '病人'

# 對訓練與測試集 case 進行分類
train_labels_new = case_train_new.apply(classify_patient)
test_labels_new = case_test_new.apply(classify_patient)

# 建立訓練與測試集 DataFrame，含特徵與 case 與身分類別
X_train_labeled = X_train_new.copy()
X_train_labeled['case'] = case_train_new.values
X_train_labeled['身分類別'] = train_labels_new.values

X_test_labeled = X_test_new.copy()
X_test_labeled['case'] = case_test_new.values
X_test_labeled['身分類別'] = test_labels_new.values

# 整體資料身分類別（不改變原資料）
all_case_new = new_df['case'].astype(str)
all_labels_new = all_case_new.apply(classify_patient)

# 印出分布統計
print("🔹 整體資料集中健康人與病人總數分布：")
print(all_labels_new.value_counts())

print("🔹 訓練集健康人及病人分布：")
print(pd.Series(train_labels_new).value_counts())
print()

print("🔹 測試集健康人及病人分布：")
print(pd.Series(test_labels_new).value_counts())
print()


### 自定義

In [ ]:
# 自訂 Pearson correlation 作為評估指標
from sklearn.metrics import make_scorer

def pearson_r(y_true, y_pred):
    return pearsonr(y_true, y_pred)[0]  # 傳回 correlation 係數（r）

# 包裝成 sklearn 可接受的 scorer（greater_is_better=True 代表數值越高越好）
pearson_scorer = make_scorer(pearson_r, greater_is_better=True)


# Random Forest

In [ ]:
"""建立 Random Forest Regressor 模型"""
rf_model = RandomForestRegressor(
    n_estimators= 300, # 100 
    max_depth= 10, # 10
    min_samples_split= 5  , # 5
    min_samples_leaf= 3,  # 3
    max_features= 'auto', 
    random_state= 42, # 42
)

"""訓練模型"""
rf_model.fit(X_train, y_train)

"""預測：訓練集"""
y_train_pred = rf_model.predict(X_train)

"""預測：測試集"""
y_test_pred = rf_model.predict(X_test)

"""訓練集 評估指標"""
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
mae_train = mean_absolute_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)
adj_r2_train = 1 - (1 - r2_train) * (len(y_train) - 1) / (len(y_train) - X_train.shape[1] - 1)
pearson_r_train = pearson_r(y_train, y_train_pred)
pearson_p_train = pearsonr(y_train, y_train_pred)[1]  # 取得 p-value

"""測試集-評估指標"""
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)
adj_r2_test = 1 - (1 - r2_test) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)
pearson_r_test = pearson_r(y_test, y_test_pred)
pearson_p_test = pearsonr(y_test, y_test_pred)[1]  # 取得 p-value

"""顯示結果"""
print("===== RF 訓練集 (Train Set) 評估 =====")

print(f"Train MSE: {mse_train:.3f}")
print(f"Train RMSE: {rmse_train:.3f}")
print(f"Train MAE: {mae_train:.3f}")
print(f"Train R²: {r2_train:.3f}")
print(f"Train Adjusted R²: {adj_r2_train:.3f}")
print(f"Train Pearson r: {pearson_r_train:.3f}")
print(f"Train Pearson p-value: {pearson_p_train:.3f}\n")

print("===== RF 測試集 (Test Set) 評估 =====")
print(f"Test MSE: {mse_test:.3f}")
print(f"Test RMSE: {rmse_test:.3f}")
print(f"Test MAE: {mae_test:.3f}")
print(f"Test R²: {r2_test:.3f}")
print(f"Test Adjusted R²: {adj_r2_test:.3f}")
print(f"Test Pearson r: {pearson_r_test:.3f}")
print(f"Test Pearson p-value: {pearson_p_test:.3f}\n")


# 計算實際值的標準差 
std_dev_train = np.std(y_train)
lower_bound = y_train - 1 * std_dev_train
upper_bound = y_train + 1 * std_dev_train
print("std_dev_train:",std_dev_train)

"""畫圖：訓練集"""
plt.figure(figsize=(4, 4))
plt.scatter(y_train, y_train_pred, color='blue', alpha=0.6, label='Train')
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'k--', label='Ideal')
# 畫出 ±2σ 的容忍區間
x_range = np.linspace(y_train.min(), y_train.max(), 100)
# plt.plot(x_range, x_range + 1 * std_dev_train, 'r--', label='+1σ')
# plt.plot(x_range, x_range - 1 * std_dev_train, 'r--', label='-1σ')
# plt.fill_between(x_range, x_range - 1 * std_dev_train, x_range + 1 * std_dev_train, color='red', alpha=0.1, label='Tolerance ±2σ')

plt.title('Random Forest - Training Set')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.legend()
plt.tight_layout()
plt.show()

# 計算實際值的標準差 (這裡以 y_test 為例)
std_dev_test = np.std(y_test)
lower_bound = y_test - 1 * std_dev_test
upper_bound = y_test + 1 * std_dev_test
print("std_dev_test:", std_dev_test)

# 畫圖：測試集（含 ±2σ 容忍區間）
plt.figure(figsize=(4, 4))
plt.scatter(y_test, y_test_pred, color='green', alpha=0.6, label='Test')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', label='Ideal')

# 畫出 ±2σ 的容忍區間
x_range = np.linspace(y_test.min(), y_test.max(), 100)
# plt.plot(x_range, x_range + 1 * std_dev_test, 'r--', label='+1σ')
# plt.plot(x_range, x_range - 1 * std_dev_test, 'r--', label='-1σ')
# plt.fill_between(x_range, x_range - 1 * std_dev_test, x_range + 1 * std_dev_test, color='red', alpha=0.1, label='Tolerance ±2σ')

plt.title('Random Forest - Testing Set')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.legend()
plt.tight_layout()
plt.show()

### 新 移動平均

In [ ]:
# 將所有資料集放入模型進行預測
all_results = pd.DataFrame({
    'case': identifiers['case'],  # 確保這列不參與數值計算
    'seg': identifiers['segment'],  # 確保這列不參與數值計算
    'true': y,
    'pred': rf_model.predict(X),
})

# 按 case 和 seg 排序
all_results = all_results.sort_values(by=['case', 'seg']).reset_index(drop=True)

# 自定義移動平均函數
def custom_moving_average(values):
    """
    計算自定義的移動平均值。
    values: 一個數值列表或 numpy array。
    返回: 一個包含移動平均值的列表。
    """
    moving_averages = []
    for i, value in enumerate(values):
        if i == 0:
            moving_averages.append(value)  # 第一個值直接加入
        else:
            new_average = (moving_averages[-1] + value) / 2  # 後續值基於前一次的平均值計算
            moving_averages.append(new_average)
    return moving_averages

# 按 case 編號進行移動平均
all_results['true_avg'] = all_results.groupby('case')['true'].transform(custom_moving_average)
all_results['pred_avg'] = all_results.groupby('case')['pred'].transform(custom_moving_average)

# 保留每個 case 的最後一筆移動平均值
final_results = all_results.groupby('case').tail(1).reset_index(drop=True)

# 計算評估指標
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr

mse_avg = mean_squared_error(final_results['true_avg'], final_results['pred_avg'])
rmse_avg = np.sqrt(mse_avg)
mae_avg = mean_absolute_error(final_results['true_avg'], final_results['pred_avg'])
r2_avg = r2_score(final_results['true_avg'], final_results['pred_avg'])
adj_r2_avg = 1 - (1 - r2_avg) * (len(final_results) - 1) / (len(final_results) - final_results.shape[1] - 1)
pearson_r_avg, p_value_avg = pearsonr(final_results['true_avg'], final_results['pred_avg'])

# 顯示結果
print("=== 移動平均後的評估指標 ===")
print(f"MSE: {mse_avg:.4f}")
print(f"RMSE: {rmse_avg:.4f}")
print(f"MAE: {mae_avg:.4f}")
print(f"R²: {r2_avg:.4f}")
print(f"Pearson r: {pearson_r_avg:.4f} (p = {p_value_avg:.4e})")

# 繪製散佈圖
plt.figure(figsize=(5, 5))
sns.regplot(x=final_results['true_avg'], 
            y=final_results['pred_avg'], 
            scatter_kws={'alpha': 0.6}, 
            line_kws={'color': 'red'})

# 加入 45 度線
min_val = min(final_results['true_avg'].min(), final_results['pred_avg'].min())
max_val = max(final_results['true_avg'].max(), final_results['pred_avg'].max())
plt.plot([min_val, max_val], [min_val, max_val], color='gray', linestyle='--', linewidth=1.2, label='45° line')

plt.title(f"Average Pearson r = {pearson_r_avg:.4f} (p = {p_value_avg:.4e})")
plt.xlabel("Average True Values")
plt.ylabel("Average Predicted Values")
plt.grid(True)
plt.tight_layout()
plt.show()

# 若需要將結果存回 CSV 檔案
output_csv_name = 'true_pred_avg_rf.csv'
final_results.to_csv(output_csv_name, index=False, encoding='utf-8-sig')
print(f"結果已輸出至：{output_csv_name}")

### 五折交叉驗證

In [ ]:
"""五折交叉驗證(輸出每一折的所有評估指標結果)"""
cv = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
r2_scores = []
results = []

for train_index, test_index in cv.split(X): # 所有資料
    # 正確用法：使用 iloc 依照列編號取出資料
    X_train_cv = X.iloc[train_index]
    X_test_cv = X.iloc[test_index]
    y_train_cv = y.iloc[train_index]
    y_test_cv = y.iloc[test_index]

    # 模型訓練
    rf_model.fit(X_train_cv, y_train_cv)

    # 預測
    y_train_pred = rf_model.predict(X_train_cv)
    y_test_pred = rf_model.predict(X_test_cv)

    # 訓練集評估
    mse_train = mean_squared_error(y_train_cv, y_train_pred)
    rmse_train = np.sqrt(mse_train)
    mae_train = mean_absolute_error(y_train_cv, y_train_pred)
    r2_train = r2_score(y_train_cv, y_train_pred) 
    adj_r2_train = 1 - (1 - r2_train) * (len(y_train_cv) - 1) / (len(y_train_cv) - X_train_cv.shape[1] - 1)
    pearson_r_train = pearson_r(y_train_cv, y_train_pred)
    p_value_train = pearsonr(y_train_cv, y_train_pred)[1]  # 取得 p-value

    # 測試集評估
    mse_test = mean_squared_error(y_test_cv, y_test_pred)
    rmse_test = np.sqrt(mse_test)
    mae_test = mean_absolute_error(y_test_cv, y_test_pred)
    r2_test = r2_score(y_test_cv, y_test_pred) 
    adj_r2_test = 1 - (1 - r2_test) * (len(y_test_cv) - 1) / (len(y_test_cv) - X_test_cv.shape[1] - 1)
    pearson_r_test = pearson_r(y_test_cv, y_test_pred)
    p_value_test = pearsonr(y_test_cv, y_test_pred)[1]  # 取得 p-value

    # 輸出每折結果
    print(f"===== RF Fold {fold} =====")
    print(f"[Train] \n MSE: {mse_train:.4f} \n RMSE: {rmse_train:.4f} \n MAE: {mae_train:.4f} \n R²: {r2_train:.4f} \n Adj R²: {adj_r2_train:.4f} \n Pearson r: {pearson_r_train:.4f}\n Pearson p-value: {p_value_train:.4f}\n")
    print(f"\n [Test ] \n MSE: {mse_test:.4f} \n RMSE: {rmse_test:.4f} \n MAE: {mae_test:.4f} \n R²: {r2_test:.4f} \n Adj R²: {adj_r2_test:.4f}\n Pearson r: {pearson_r_test:.4f}\n Pearson p-value: {p_value_test:.4f}\n")

    # 可選擇儲存所有指標到列表中以利後續統計分析
    results.append({
        "Fold": fold,
        "Train_MSE": mse_train.round(3),
        "Train_RMSE": rmse_train.round(3),
        "Train_MAE": mae_train.round(3),
        "Train_R2": r2_train.round(3),
        "Train_Adj_R2": adj_r2_train.round(3),
        "Train_Pearson_r": pearson_r_train.round(3),
        "Train_p_value": pearson_p_train.round(3),
        
        "Test_MSE": mse_test.round(2),
        "Test_RMSE": rmse_test.round(2),
        "Test_MAE": mae_test.round(2),
        "Test_R2": r2_test,
        "Test_Adj_R2": adj_r2_test,
        "Test_Pearson_r": pearson_r_test.round(3),
        "Test_p_value": pearson_p_test.round(3)
    })
    
    # 建立左右並排的子圖
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))  # 1列2欄子圖

    # 畫圖：訓練集 True vs Predicted
    axes[0].scatter(y_train_cv, y_train_pred, alpha=0.7, edgecolors='g')
    axes[0].plot([y_train_cv.min(), y_train_cv.max()], [y_train_cv.min(), y_train_cv.max()], 'r--')
    axes[0].set_title(f"Fold {fold} - Train")
    axes[0].set_xlabel("True Values")
    axes[0].set_ylabel("Predicted Values")
    axes[0].grid(True)

    # 畫圖：測試集 True vs Predicted
    axes[1].scatter(y_test_cv, y_test_pred, alpha=0.7, edgecolors='b')
    axes[1].plot([y_test_cv.min(), y_test_cv.max()], [y_test_cv.min(), y_test_cv.max()], 'r--')
    axes[1].set_title(f"Fold {fold} - Test")
    axes[1].set_xlabel("True Values")
    axes[1].set_ylabel("Predicted Values")
    axes[1].grid(True)

    # 整體排版
    plt.suptitle(f"Fold {fold} - True vs. Predicted")
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # 預留上方空間給 suptitle
    plt.show()

    fold += 1
    
# 將結果轉成 DataFrame
results_df = pd.DataFrame(results)

# 顯示每個指標的平均與標準差（分訓練與測試集）
mean_results = results_df.mean(numeric_only=True)
std_results = results_df.std(numeric_only=True)

print("===== 指標平均與標準差 (5折) =====")
for col in results_df.columns:
    if col != "Fold":
        mean_val = mean_results[col]
        std_val = std_results[col]
        print(f"{col}: 平均 = {mean_val:.4f}，標準差 = {std_val:.4f}")

results_df.to_csv("cv_results_rf.csv", index=False, encoding='utf-8-sig')
print("\n已儲存每折交叉驗證結果至 'cv_results_rf.csv'")

# 橫向格式 summary
summary_df = pd.DataFrame({
    metric: [f"{mean:.2f}({std:.2f})"]
    for metric, mean, std in zip(mean_results.index, mean_results, std_results)
})

# 選擇性：加入模型名稱作為 row index
summary_df.insert(0, 'Model', 'AdaBoost')  # ← 若是其他模型名稱也可更改

# 顯示結果
print("\n===== 五折交叉驗證統計（橫向 mean(std) 格式）=====")
print(summary_df)

# 儲存為 CSV
summary_df.to_csv("cv_summary_rf.csv", index=False, encoding='utf-8-sig')
print("✅ 已儲存橫向 mean(std) 格式至 'cv_summary_rf.csv'")

### 移動平均後的Pearson相關係數

In [ ]:
# 將預測結果與識別欄位合併
results = pd.DataFrame({
    'case': identifiers_test['case'],
    'seg': identifiers_test['segment'],
    'true': y_test,
    'pred': y_test_pred,
})

# 按 case 和 seg 排序
results = results.sort_values(by=['case', 'seg']).reset_index(drop=True)

# 自定義移動平均函數
def custom_moving_average(values):
    """
    計算自定義的移動平均值。
    values: 一個數值列表或 numpy array。
    返回: 一個包含移動平均值的列表。
    """
    moving_averages = []
    for i, value in enumerate(values):
        if i == 0:
            # 第一個值直接加入
            moving_averages.append(value)
        else:
            # 後續值基於前一次的平均值計算
            new_average = (moving_averages[-1] + value) / 2
            moving_averages.append(new_average)
    return moving_averages

# 8. 按 case 編號進行移動平均（使用自定義函數）
results['true_avg'] = results.groupby('case')['true'].transform(custom_moving_average)
results['pred_avg'] = results.groupby('case')['pred'].transform(custom_moving_average)

# results['true_avg'] = results.groupby('case')['true'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
# results['pred_avg'] = results.groupby('case')['pred'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

# 保留每個 case 的最後一筆移動平均值（可選）
final_results = results.groupby('case').tail(1).reset_index(drop=True)

# 若需要將結果存回 CSV 檔案
output_csv_name = 'true_pred_rf.csv'
results.to_csv(output_csv_name, index=False,encoding='utf-8-sig')
print(f"結果已輸出至：{output_csv_name}")

#=====================================================
# 計算 Pearson 相關係數
pearson_r_avg_test, p_value_avg_test = pearsonr(final_results['true_avg'], final_results['pred_avg'])
print(f"Test true_pred_avg Pearson r: {pearson_r_avg_test:.4f} (p={p_value_avg_test:.4f})")

# 繪製散佈圖
plt.figure(figsize=(5, 5))
sns.regplot(x=final_results['true_avg'], 
            y=final_results['pred_avg'], 
            scatter_kws={'alpha': 0.6}, 
            line_kws={'color': 'red'})

# 加入 45 度線
min_val = min(final_results['true_avg'].min(), final_results['pred_avg'].min())
max_val = max(final_results['true_avg'].max(), final_results['pred_avg'].max())
plt.plot([min_val, max_val], [min_val, max_val], color='gray', linestyle='--', linewidth=1.2, label='45° line')

plt.title(f"Average Pearson r = {pearson_r_avg_test:.4f} (p = {p_value_avg_test:.4e})")
plt.xlabel("Average True Values")
plt.ylabel("Average Predicted Values")
plt.grid(True)
plt.tight_layout()
plt.show()


### 所有資料移動平均後效果

In [ ]:
# 确保特征列只包含数值数据
identifier_columns = ['case', 'segment']
feature_columns = [col for col in df.columns if col not in identifier_columns + ['hr']]

X = df_pre[feature_columns]
X = X.apply(pd.to_numeric, errors='coerce')  # 转换为数值类型
X = X.dropna(axis=1)  # 删除无法转换的列

y = df_pre['hr']

# 切分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 检查数据类型
# print(X_train.dtypes)
# print(X_test.dtypes)

# 确保模型训练和预测使用数值数据
rf_model.fit(X_train, y_train)
y_test_pred = rf_model.predict(X_test)

# 创建结果 DataFrame
all_results = pd.DataFrame({
    'case': identifiers_test['case'],  # 确保这列不会参与数值计算
    'seg': identifiers_test['segment'],  # 确保这列不会参与数值计算
    'true': y_test,
    'pred': y_test_pred,
})

# 按 case 和 seg 排序
all_results = all_results.sort_values(by=['case', 'seg']).reset_index(drop=True)

# 自定義指數型移動平均函數
def exponential_moving_average(values, alpha=0.5):
    """
    計算指數型移動平均值。
    values: 一個數值列表或 numpy array。
    alpha: 平滑係數，範圍在 0 到 1。
    返回: 一個包含指數型移動平均值的列表。
    """
    ema_values = []
    for i, value in enumerate(values):
        if i == 0:
            ema_values.append(value)  # 第一個值直接加入
        else:
            ema_values.append(alpha * value + (1 - alpha) * ema_values[-1])
    return ema_values

# 按 case 編號進行指數型移動平均
all_results['true_ema'] = all_results.groupby('case')['true'].transform(lambda x: exponential_moving_average(x, alpha=0.5))
all_results['pred_ema'] = all_results.groupby('case')['pred'].transform(lambda x: exponential_moving_average(x, alpha=0.5))

# 保留每個 case 的最後一筆指數型移動平均值（可選）
final_ema_results = all_results.groupby('case').tail(1).reset_index(drop=True)

# 計算評估指標
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 計算移動平均前的評估指標
mse_before = mean_squared_error(all_results['true'], all_results['pred'])
rmse_before = np.sqrt(mse_before)
mae_before = mean_absolute_error(all_results['true'], all_results['pred'])
r2_before = r2_score(all_results['true'], all_results['pred'])
pearson_r_before, p_value_before = pearsonr(all_results['true'], all_results['pred'])

# 顯示移動平均前的評估指標
print("=== 移動平均前的效果 ===")
print(f"MSE: {mse_before:.4f}")
print(f"RMSE: {rmse_before:.4f}")
print(f"MAE: {mae_before:.4f}")
print(f"R²: {r2_before:.4f}")
print(f"Adjusted R²: {1 - (1 - r2_before) * (len(all_results) - 1) / (len(all_results) - all_results.shape[1] - 1):.4f}")
print(f"Pearson r: {pearson_r_before:.4f} (p = {p_value_before:.4e})")
print(f"数据点数量（移动平均前）: {len(all_results)}")

# 繪製移動平均前的散佈圖
plt.figure(figsize=(5, 5))
sns.regplot(x=all_results['true'], 
            y=all_results['pred'], 
            scatter_kws={'alpha': 0.6}, 
            line_kws={'color': 'red'})

# 加入 45 度線
min_val_before = min(all_results['true'].min(), all_results['pred'].min())
max_val_before = max(all_results['true'].max(), all_results['pred'].max())
plt.plot([min_val_before, max_val_before], [min_val_before, max_val_before], color='gray', linestyle='--', linewidth=1.2, label='45° line')

plt.title(f"Before Moving Average Pearson r = {pearson_r_before:.4f} (p = {p_value_before:.4e})")
plt.xlabel("True Values")
plt.ylabel("Predicted Values")
plt.grid(True)
plt.tight_layout()
plt.show()

# 計算移動平均後的評估指標
mse_after = mean_squared_error(final_ema_results['true_ema'], final_ema_results['pred_ema'])
rmse_after = np.sqrt(mse_after)
mae_after = mean_absolute_error(final_ema_results['true_ema'], final_ema_results['pred_ema'])
r2_after = r2_score(final_ema_results['true_ema'], final_ema_results['pred_ema'])
pearson_r_after, p_value_after = pearsonr(final_ema_results['true_ema'], final_ema_results['pred_ema'])

# 顯示移動平均後的評估指標
print("\n=== 移動平均後的效果 ===")
print(f"MSE: {mse_after:.4f}")
print(f"RMSE: {rmse_after:.4f}")
print(f"MAE: {mae_after:.4f}")
print(f"R²: {r2_after:.4f}")
print(f"Adjusted R²: {1 - (1 - r2_after) * (len(final_ema_results) - 1) / (len(final_ema_results) - final_ema_results.shape[1] - 1):.4f}")
print(f"Pearson r: {pearson_r_after:.4f} (p = {p_value_after:.4e})")
print(f"数据点数量（移动平均后）: {len(final_ema_results)}")

# 繪製移動平均後的散佈圖
plt.figure(figsize=(5, 5))
sns.regplot(x=final_ema_results['true_ema'], 
            y=final_ema_results['pred_ema'], 
            scatter_kws={'alpha': 0.6}, 
            line_kws={'color': 'red'})

# 加入 45 度線
min_val_after = min(final_ema_results['true_ema'].min(), final_ema_results['pred_ema'].min())
max_val_after = max(final_ema_results['true_ema'].max(), final_ema_results['pred_ema'].max())
plt.plot([min_val_after, max_val_after], [min_val_after, max_val_after], color='gray', linestyle='--', linewidth=1.2, label='45° line')

plt.title(f"Exponential Moving Average Pearson r = {pearson_r_after:.4f} (p = {p_value_after:.4e})")
plt.xlabel("Exponential Moving Average True Values")
plt.ylabel("Exponential Moving Average Predicted Values")
plt.grid(True)
plt.tight_layout()
plt.show()

### 輸出csv

In [ ]:
"""輸出csv，參數、模型評估指標、pearson 相關係數"""
model_params = (
    f"n_estimators={rf_model.n_estimators}, "
    f"max_depth={rf_model.max_depth}, "
    f"min_samples_split={rf_model.min_samples_split}, "
    f"min_samples_leaf={rf_model.min_samples_leaf}, "
    f"max_features={rf_model.max_features}, "
    f"random_state={rf_model.random_state}"
)

# 提取文件名（去掉路径，只保留文件名）
file_name = os.path.basename(path)

# 建立結果字典
results_dict = {
    "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),  # 加入時間戳記
    "Model": "Random Forest Regressor",
    "path": file_name,
    "y":y.name,
    "Train_MSE": round(mse_train,3),
    "Train_RMSE": round(rmse_train,3),
    "Train_MAE": round(mae_train,3),
    "Train_R2": round(r2_train,3),
    "Train_Adjusted_R2": round(adj_r2_train,3),
    "Train_Pearson_r": round(pearson_r_train, 3),
    "Train_p_value": round(p_value_train, 3),

    "Test_MSE": round(mse_test,3),
    "Test_RMSE": round(rmse_test,3),
    "Test_MAE": round(mae_test,3),
    "Test_R2": round(r2_test,3),
    "Test_Adjusted_R2": round(adj_r2_test,3),
    "Test_Pearson_r": round(pearson_r_test, 3),
    "Train_p_value": round(p_value_test, 3),

    "Avg_test_Pearson_r": round(pearson_r_avg_test, 3),
    "Avg_test_Pearson_p": round(p_value_avg_test, 3),
    "model_params": model_params,
}

# 將結果轉換為 DataFrame
results_df = pd.DataFrame([results_dict])

# 輸出檔案名稱
output_csv_name = "train_test_rf.csv"

# 若檔案已存在，則附加資料；否則建立新檔案
if os.path.exists(output_csv_name):
    results_df.to_csv(output_csv_name, mode='a', header=False, index=False)
else:
    results_df.to_csv(output_csv_name, index=False)

print(f"結果已輸出至：{output_csv_name}")

### Shapley Value

In [ ]:
import shap
import matplotlib.pyplot as plt
import pandas as pd

"""Permutation SHAP"""
# # 回歸模型預測函數：直接使用 predict() 回傳連續數值
# rf_predict_fn = lambda x: rf_model.predict(pd.DataFrame(x, columns=X_test.columns))

# # 建立 explainer 並取得 SHAP 值（回歸問題）
# explainer_rf = shap.PermutationExplainer(rf_predict_fn, X_test)
# shap_values_rf = explainer_rf(X_test)

"""tree SHAP"""
explainer_rf = shap.Explainer(rf_model, X_test) 
shap_values_rf = explainer_rf(X_test)

# Random Forest
top_n = 15
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_rf.values, X_test, show=False, max_display= top_n)
# plt.title("Random Forest Feature Importance (Permutation SHAP)")
plt.title("Random Forest Feature Importance (Tree SHAP)")
plt.show()


### 特徵重要性分析

In [ ]:
"""特徵重要性分析"""
importances = rf_model.feature_importances_  # 取得特徵重要性
feature_names = X_train.columns              # 特徵名稱
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# 顯示 Top 10
print("\n===== 特徵重要性 (Top 20) =====")
print(importance_df.head(20))

# 畫圖：Top 10
plt.figure(figsize=(10, 10))
plt.barh(importance_df['Feature'][:20][::-1], importance_df['Importance'][:20][::-1], color='steelblue')
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.tight_layout()
plt.show()

### 貝氏優化

In [ ]:
"""Random Forest 貝氏優化"""
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from skopt import BayesSearchCV
from skopt.space import Integer
import numpy as np
from sklearn.metrics import make_scorer
from scipy.stats import pearsonr



# 建立回歸模型
# model = RandomForestRegressor(random_state=42)

# 自訂 Pearson correlation 作為評估指標
# def pearson_r(y_true, y_pred):
#     return pearsonr(y_true, y_pred)[0]  # 傳回 correlation 係數（r）

# # 包裝成 sklearn 可接受的 scorer（greater_is_better=True 代表數值越高越好）
# pearson_scorer = make_scorer(pearson_r, greater_is_better=True)

"""建立 Random Forest Regressor 模型"""
# rf_model = RandomForestRegressor(
#     n_estimators= 300, # 100 
#     max_depth= 10, # 10
#     min_samples_split= 5  , # 5
#     min_samples_leaf= 3,  # 3
#     max_features= 'auto', 
#     random_state= 42, # 42
# )

# 定義貝式優化的超參數搜尋空間
param_space = {
    'n_estimators': Integer(100, 500),
    'max_depth': Integer(3, 15),
    # 'min_samples_split': Integer(3, 10),
    # 'min_samples_leaf': Integer(3, 10),
    'max_features': Integer(1, X_train.shape[1]),  # 1 ~ 特徵數量
    # 'max_features': Integer(1, 50)
}

# 執行 BayesSearchCV
bayes_search = BayesSearchCV(
    estimator= rf_model,            # !!記得要改！要優化的模型
    search_spaces=param_space,  # 超參數的搜尋空間
    n_iter=100,                
    cv=5,                      # 幾折交叉驗證
    scoring='r2',  
    random_state=42,
    #n_jobs=-1,               # 使用的 CPU 核心數（-1 為全部)
    # verbose=1                # 顯示訓練過程（1 為基本）
)

# 用訓練資料幫你自動找出「最佳參數組合」
bayes_search.fit(X_train, y_train)

# 取得最佳參數和分數
print("最佳參數:")
print(bayes_search.best_params_)


# 直接使用最佳模型做預測（推薦）
best_model = bayes_search.best_estimator_ # 取出已訓練好的最佳模型

# 在測試集上預測
y_test_pred = best_model.predict(X_test) # 用測試集做預測

mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)
adj_r2 = 1 - (1 - r2) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)
pearson = pearsonr(y_test, y_test_pred)[0]
p_value = pearsonr(y_test, y_test_pred)[1]  # 取得 p-value

print(f"測試集結果：")
print(f"MSE:{mse:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"R²: {r2:.3f}")
print(f"adjusted r2:{adj_r2:.3f}")
print(f"Pearson r: {pearson:.3f}")
print(f"Pearson p-value: {p_value:.3f}")

# 計算交叉驗證 R² 分數
cv_scores = cross_val_score(bayes_search.best_estimator_, X_train, y_train, cv=5, scoring='r2')
mean_score = np.mean(cv_scores)
std_score = np.std(cv_scores)

print(f"最佳交叉驗證r2: {mean_score:.4f} ± {std_score:.4f}")

### PCA

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. 使用 PCA 降維訓練資料
pca = PCA(n_components=0.9)  # 保留90%變異量
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)  # 測試資料也要 transform！

print("主成分數量：", pca.n_components_)
print("各個特徵變異量：", pca.explained_variance_ratio_)
print("保留的總變異量：", pca.explained_variance_ratio_.sum())


# 2. 繪製主成分貢獻度圖（可選）
try:
    feature_names = X_train.columns.tolist()
except:
    feature_names = [f"Feature {i}" for i in range(X_train.shape[1])]

num_components_to_plot = pca.n_components_  # 顯示前幾個主成分
top_n_features = 10         # 每個主成分顯示前幾名貢獻特徵

plt.figure(figsize=(15, 5 * num_components_to_plot))

for i in range(num_components_to_plot):
    # 取得該主成分對所有特徵的權重
    component_weights = pca.components_[i]
    
    # 根據貢獻度絕對值排序，取得前 top_n 名的索引
    top_indices = np.argsort(np.abs(component_weights))[::-1][:top_n_features]
    
    # 取得對應特徵名稱與權重
    top_features = [feature_names[j] for j in top_indices]
    top_weights = component_weights[top_indices]

    # 繪製橫條圖
    plt.figure(figsize=(8, 4))
    bars = plt.barh(top_features, top_weights, color='skyblue')
    plt.title(f"Top {top_n_features} Contributing Features to PC{i+1}")
    plt.xlabel("Contribution Weight")
    plt.ylabel("Feature")
    plt.grid(True, axis='x')

    # 加上數值標籤
    for bar in bars:
        width = bar.get_width()
        plt.text(width, bar.get_y() + bar.get_height()/2, f'{width:.2f}', va='center')

    plt.tight_layout()
    plt.show()


### PCA後測試

In [ ]:
rf_model.fit(X_train_pca, y_train)

# --- 預測 ---
y_train_pred = rf_model.predict(X_train_pca)
y_pred = rf_model.predict(X_test_pca)

# --- 訓練集評估 ---
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
mae_train = mean_absolute_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)
adjusted_r2_train = 1 - (1 - r2_train) * (len(y_train) - 1) / (len(y_train) - X_train_pca.shape[1] - 1)

print("\n=== PCA主成分-訓練集評估===")
print("MSE:", mse_train)
print("RMSE:", rmse_train)
print("MAE:", mae_train)
print("R²:", r2_train)
print("Adjusted R²:", adjusted_r2_train)

# --- 測試集評估 ---
mse_test = mean_squared_error(y_test, y_pred)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(y_test, y_pred)
r2_test = r2_score(y_test, y_pred)
adjusted_r2_test = 1 - (1 - r2_test) * (len(y_test) - 1) / (len(y_test) - X_test_pca.shape[1] - 1)

print("\n=== PCA主成分-測試集評估===")
print("MSE:", mse_test)
print("RMSE:", rmse_test)
print("MAE:", mae_test)
print("R²:", r2_test)
print("Adjusted R²:", adjusted_r2_test)

# AdaBoost

In [ ]:
"""建立 AdaBoost 模型"""
ada_model = AdaBoostRegressor(
    base_estimator= DecisionTreeRegressor(max_depth=8),
    n_estimators=300,
    learning_rate=0.1,
    random_state=42
)

ada_model.fit(X_train, y_train) # 訓練模型
y_train_pred = ada_model.predict(X_train) # 預測：訓練集
y_test_pred = ada_model.predict(X_test) # 預測：測試集


"""訓練集 評估指標"""
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
mae_train = mean_absolute_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)
adj_r2_train = 1 - (1 - r2_train) * (len(y_train) - 1) / (len(y_train) - X_train.shape[1] - 1)
pearson_r_train = pearson_r(y_train, y_train_pred)
pearson_p_train = pearsonr(y_train, y_train_pred)[1]  # 取得 p-value

"""測試集-評估指標"""
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)
adj_r2_test = 1 - (1 - r2_test) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)
pearson_r_test = pearson_r(y_test, y_test_pred)
pearson_p_test = pearsonr(y_test, y_test_pred)[1]  # 取得 p-value

"""顯示結果"""
print("===== AdaBoost 訓練集 (Train Set) 評估 =====")

print(f"Train MSE: {mse_train:.3f}")
print(f"Train RMSE: {rmse_train:.3f}")
print(f"Train MAE: {mae_train:.3f}")
print(f"Train R²: {r2_train:.3f}")
print(f"Train Adjusted R²: {adj_r2_train:.3f}")
print(f"Train Pearson r: {pearson_r_train:.3f}")
print(f"Train Pearson p-value: {pearson_p_train:.3f}\n")

print("===== AdaBoost 測試集 (Test Set) 評估 =====")
print(f"Test MSE: {mse_test:.3f}")
print(f"Test RMSE: {rmse_test:.3f}")
print(f"Test MAE: {mae_test:.3f}")
print(f"Test R²: {r2_test:.3f}")
print(f"Test Adjusted R²: {adj_r2_test:.3f}")
print(f"Test Pearson r: {pearson_r_test:.3f}")
print(f"Test Pearson p-value: {pearson_p_test:.3f}\n")

# 計算實際值的標準差 
std_dev_train = np.std(y_train)
lower_bound = y_train - 1 * std_dev_train
upper_bound = y_train + 1 * std_dev_train
print("std_dev_train:",std_dev_train)

"""畫圖：訓練集"""
plt.figure(figsize=(4, 4))
plt.scatter(y_train, y_train_pred, color='blue', alpha=0.6, label='Train')
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'k--', label='Ideal')
# 畫出 ±2σ 的容忍區間
x_range = np.linspace(y_train.min(), y_train.max(), 100)
# plt.plot(x_range, x_range + 1 * std_dev_train, 'r--', label='+1σ')
# plt.plot(x_range, x_range - 1 * std_dev_train, 'r--', label='-1σ')
# plt.fill_between(x_range, x_range - 1 * std_dev_train, x_range + 1 * std_dev_train, color='red', alpha=0.1, label='Tolerance ±2σ')

plt.title('AdaBoost - Training Set')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.legend()
plt.tight_layout()
plt.show()

# 計算實際值的標準差 (這裡以 y_test 為例)
std_dev_test = np.std(y_test)
lower_bound = y_test - 1 * std_dev_test
upper_bound = y_test + 1 * std_dev_test
print("std_dev_test:", std_dev_test)

"""畫圖：訓練集"""
plt.figure(figsize=(4, 4))
plt.scatter(y_test, y_test_pred, color='green', alpha=0.6, label='Test')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', label='Ideal')

# 畫出 ±2σ 的容忍區間
x_range = np.linspace(y_test.min(), y_test.max(), 100)
# plt.plot(x_range, x_range + 1 * std_dev_test, 'r--', label='+1σ')
# plt.plot(x_range, x_range - 1 * std_dev_test, 'r--', label='-1σ')
# plt.fill_between(x_range, x_range - 1 * std_dev_test, x_range + 1 * std_dev_test, color='red', alpha=0.1, label='Tolerance ±2σ')

plt.title('AdaBoost - Testing Set')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.legend()
plt.tight_layout()
plt.show()

### 五折交叉驗證

In [ ]:
"""五折交叉驗證(輸出每一折的所有評估指標結果)-ADA-"""
cv = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
r2_scores = []
results = []

for train_index, test_index in cv.split(X):
    # 正確用法：使用 iloc 依照列編號取出資料
    X_train_cv = X.iloc[train_index]
    X_test_cv = X.iloc[test_index]
    y_train_cv = y.iloc[train_index]
    y_test_cv = y.iloc[test_index]

    # 模型訓練
    ada_model.fit(X_train_cv, y_train_cv)

    # 預測
    y_train_pred = ada_model.predict(X_train_cv)
    y_test_pred = ada_model.predict(X_test_cv)

    # 訓練集評估
    mse_train = mean_squared_error(y_train_cv, y_train_pred)
    rmse_train = np.sqrt(mse_train)
    mae_train = mean_absolute_error(y_train_cv, y_train_pred)
    r2_train = r2_score(y_train_cv, y_train_pred) 
    adj_r2_train = 1 - (1 - r2_train) * (len(y_train_cv) - 1) / (len(y_train_cv) - X_train_cv.shape[1] - 1)
    pearson_r_train = pearson_r(y_train_cv, y_train_pred)
    p_value_train = pearsonr(y_train_cv, y_train_pred)[1]  # 取得 p-value

    # 測試集評估
    mse_test = mean_squared_error(y_test_cv, y_test_pred)
    rmse_test = np.sqrt(mse_test)
    mae_test = mean_absolute_error(y_test_cv, y_test_pred)
    r2_test = r2_score(y_test_cv, y_test_pred) 
    adj_r2_test = 1 - (1 - r2_test) * (len(y_test_cv) - 1) / (len(y_test_cv) - X_test_cv.shape[1] - 1)
    pearson_r_test = pearson_r(y_test_cv, y_test_pred)
    p_value_test = pearsonr(y_test_cv, y_test_pred)[1]  # 取得 p-value

    # 輸出每折結果
    print(f"===== AdaBoost Fold {fold} =====")
    print(f"[Train] \n MSE: {mse_train:.4f} \n RMSE: {rmse_train:.4f} \n MAE: {mae_train:.4f} \n R²: {r2_train:.4f} \n Adj R²: {adj_r2_train:.4f} \n Pearson r: {pearson_r_train:.4f}\n Pearson p-value: {p_value_train:.4f}\n")
    print(f"\n [Test ] \n MSE: {mse_test:.4f} \n RMSE: {rmse_test:.4f} \n MAE: {mae_test:.4f} \n R²: {r2_test:.4f} \n Adj R²: {adj_r2_test:.4f}\n Pearson r: {pearson_r_test:.4f}\n Pearson p-value: {p_value_test:.4f}\n")

    # 可選擇儲存所有指標到列表中以利後續統計分析
    results.append({
        "Fold": fold,
        "Train_MSE": mse_train.round(3),
        "Train_RMSE": rmse_train.round(3),
        "Train_MAE": mae_train.round(3),
        "Train_R2": r2_train.round(3),
        "Train_Adj_R2": adj_r2_train.round(3),
        "Train_Pearson_r": pearson_r_train.round(3),
        "Train_p_value": pearson_p_train.round(3),
        
        "Test_MSE": mse_test.round(2),
        "Test_RMSE": rmse_test.round(2),
        "Test_MAE": mae_test.round(2),
        "Test_R2": r2_test,
        "Test_Adj_R2": adj_r2_test,
        "Test_Pearson_r": pearson_r_test.round(3),
        "Test_p_value": pearson_p_test.round(3)
    })
    
    # 建立左右並排的子圖
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))  # 1列2欄子圖

    # 畫圖：訓練集 True vs Predicted
    axes[0].scatter(y_train_cv, y_train_pred, alpha=0.7, edgecolors='g')
    axes[0].plot([y_train_cv.min(), y_train_cv.max()], [y_train_cv.min(), y_train_cv.max()], 'r--')
    axes[0].set_title(f"Fold {fold} - Train")
    axes[0].set_xlabel("True Values")
    axes[0].set_ylabel("Predicted Values")
    axes[0].grid(True)

    # 畫圖：測試集 True vs Predicted
    axes[1].scatter(y_test_cv, y_test_pred, alpha=0.7, edgecolors='b')
    axes[1].plot([y_test_cv.min(), y_test_cv.max()], [y_test_cv.min(), y_test_cv.max()], 'r--')
    axes[1].set_title(f"Fold {fold} - Test")
    axes[1].set_xlabel("True Values")
    axes[1].set_ylabel("Predicted Values")
    axes[1].grid(True)

    # 整體排版
    plt.suptitle(f"Fold {fold} - True vs. Predicted")
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # 預留上方空間給 suptitle
    plt.show()

    fold += 1

# 將結果轉成 DataFrame
results_df = pd.DataFrame(results)

# 顯示每個指標的平均與標準差（分訓練與測試集）
mean_results = results_df.mean(numeric_only=True)
std_results = results_df.std(numeric_only=True)

print("===== 指標平均與標準差 (5折) =====")
for col in results_df.columns:
    if col != "Fold":
        mean_val = mean_results[col]
        std_val = std_results[col]
        print(f"{col}: 平均 = {mean_val:.4f}，標準差 = {std_val:.4f}")

results_df.to_csv("cv_results_ada.csv", index=False, encoding='utf-8-sig')
print("\n已儲存每折交叉驗證結果至 'cv_results_ada.csv'")

# 橫向格式 summary
summary_df = pd.DataFrame({
    metric: [f"{mean:.2f}({std:.2f})"]
    for metric, mean, std in zip(mean_results.index, mean_results, std_results)
})

# 選擇性：加入模型名稱作為 row index
summary_df.insert(0, 'Model', 'AdaBoost')  # ← 若是其他模型名稱也可更改

# 顯示結果
print("\n===== 五折交叉驗證統計（橫向 mean(std) 格式）=====")
print(summary_df)

# 儲存為 CSV
summary_df.to_csv("cv_summary_ada.csv", index=False, encoding='utf-8-sig')
print("✅ 已儲存橫向 mean(std) 格式至 'cv_summary_ada.csv'")

### 移動平均後的Pearson相關係數

In [ ]:
"""移動平均後的Pearson r"""

# 將預測結果與識別欄位合併
results = pd.DataFrame({
    'case': identifiers_test['case'],
    'seg': identifiers_test['segment'],
    'true': y_test,
    'pred': y_test_pred,
})

# 按 case 和 seg 排序
results = results.sort_values(by=['case', 'seg']).reset_index(drop=True)

# 按 case 編號進行移動平均
results['true_avg'] = results.groupby('case')['true'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
results['pred_avg'] = results.groupby('case')['pred'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

# 保留每個 case 的最後一筆移動平均值（可選）
final_results = results.groupby('case').tail(1).reset_index(drop=True)

# 存回 CSV 檔案(實際值vs預測值)
output_csv_name = 'true_pred_ada.csv'
results.to_csv(output_csv_name, index=False,encoding='utf-8-sig')
print(f"結果已輸出至：{output_csv_name}")

#=====================================================
# 計算 Pearson 相關係數

pearson_r_avg_test, p_value_avg_test = pearsonr(final_results['true_avg'], final_results['pred_avg'])
print(f"Test true_pred_avg Pearson r: {pearson_r_avg_test:.4f} (p={p_value_avg_test:.4f})")

# 繪製散佈圖
plt.figure(figsize=(5, 5))
sns.regplot(x=final_results['true_avg'], 
            y=final_results['pred_avg'], 
            scatter_kws={'alpha': 0.6}, 
            line_kws={'color': 'red'})

# 加入 45 度線
min_val = min(final_results['true_avg'].min(), final_results['pred_avg'].min())
max_val = max(final_results['true_avg'].max(), final_results['pred_avg'].max())
plt.plot([min_val, max_val], [min_val, max_val], color='gray', linestyle='--', linewidth=1.2, label='45° line')

plt.title(f"Average Pearson r = {pearson_r_avg_test:.4f} (p = {p_value_avg_test:.4e})")
plt.xlabel("Average True Values")
plt.ylabel("Average Predicted Values")
plt.grid(True)
plt.tight_layout()
plt.show()

### 輸出csv

In [ ]:
"""輸出csv，參數、模型評估指標、pearson 相關係數"""

model_params = (
    f"base_estimator={ada_model.base_estimator}, "
    f"n_estimators={ada_model.n_estimators}, "
    f"learning_rate={ada_model.learning_rate}, "
    f"random_state={ada_model.random_state}"
)

# 提取文件名（去掉路径，只保留文件名）
file_name = os.path.basename(path)

# 建立結果字典
results_dict = {
    "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),  # 加入時間戳記
    "Model": "Random Forest Regressor",
    "path": file_name,
    "y":y.name,
    "Train_MSE": round(mse_train,3),
    "Train_RMSE": round(rmse_train,3),
    "Train_MAE": round(mae_train,3),
    "Train_R2": round(r2_train,3),
    "Train_Adjusted_R2": round(adj_r2_train,3),
    "Train_Pearson_r": round(pearson_r_train, 3),
    "Train_p_value": round(p_value_train, 3),

    "Test_MSE": round(mse_test,3),
    "Test_RMSE": round(rmse_test,3),
    "Test_MAE": round(mae_test,3),
    "Test_R2": round(r2_test,3),
    "Test_Adjusted_R2": round(adj_r2_test,3),
    "Test_Pearson_r": round(pearson_r_test, 3),
    "Train_p_value": round(p_value_test, 3),

    # "Avg_test_Pearson_r": round(pearson_r_avg_test, 3),
    # "Avg_test_Pearson_p": round(p_value_avg_test, 3),
    "model_params": model_params,
}

# 將結果轉換為 DataFrame
results_df = pd.DataFrame([results_dict])

# 輸出檔案名稱
output_csv_name = "train_test_ada.csv"

# 若檔案已存在，則附加資料；否則建立新檔案
if os.path.exists(output_csv_name):
    results_df.to_csv(output_csv_name, mode='a', header=False, index=False)
else:
    results_df.to_csv(output_csv_name, index=False)

print(f"結果已輸出至：{output_csv_name}")


### Shapley Value

In [ ]:
import shap
import matplotlib.pyplot as plt
import pandas as pd

"""Permutatio SHAP"""
# # 回歸模型預測函數：直接使用 predict() 回傳連續數值
# ada_predict_fn = lambda x: ada_model.predict(pd.DataFrame(x, columns=X_test.columns))
# # 建立 explainer 並取得 SHAP 值（回歸問題）
# explainer_ada = shap.PermutationExplainer(ada_predict_fn, X_test)
# shap_values_ada = explainer_ada(X_test)

"""tree SHAP"""
# 回歸模型預測函數：直接使用 predict() 回傳連續數值
ada_predict_fn = lambda x: ada_model.predict(pd.DataFrame(x, columns=X_test.columns))
explainer_ada = shap.Explainer(ada_predict_fn, X_test) 
shap_values_ada = explainer_ada(X_test)

top_n = 15
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_rf.values, X_test, show=False, max_display= top_n)

# plt.title("Random Forest Feature Importance (Permutation SHAP)")
plt.title("AdaBoost (Tree SHAP)")
plt.show()

### 特徵重要性分析

In [ ]:
"""特徵重要性分析"""
importances = ada_model.feature_importances_  # 取得特徵重要性
feature_names = X_train.columns              # 特徵名稱
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# 顯示 Top 10
print("\n===== 特徵重要性 (Top 20) =====")
print(importance_df.head(20))

# 畫圖：Top 10
plt.figure(figsize=(10, 10))
plt.barh(importance_df['Feature'][:20][::-1], importance_df['Importance'][:20][::-1], color='steelblue')
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances (Random Forest)')
plt.tight_layout()
plt.show()

### 貝氏優化

In [ ]:
"""AdaBoost 貝氏優化"""
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from skopt import BayesSearchCV
from skopt.space import Integer, Real
import numpy as np

# 定義貝氏優化的超參數搜尋空間
param_space = {
    'base_estimator__max_depth': Integer(5, 15),  # 基礎決策樹的最大深度
    'n_estimators': Integer(100, 500),             # 迭代次數    
    'learning_rate': Real(0.01, 0.1, prior='uniform'),  # 學習率               # 隨機種子
}

# 執行 BayesSearchCV
bayes_search = BayesSearchCV(
    estimator= ada_model,
    search_spaces=param_space,
    n_iter=100,
    cv=5,
    scoring= 'r2', 
    random_state=42,
    # n_jobs=-1,
)

# 用訓練資料幫你自動找出「最佳參數組合」
bayes_search.fit(X_train, y_train)

# 取得最佳參數和分數
print("最佳參數:")
print(bayes_search.best_params_)


# 直接使用最佳模型做預測（推薦）
best_model = bayes_search.best_estimator_ # 取出已訓練好的最佳模型

# 在測試集上預測
y_test_pred = best_model.predict(X_test) # 用測試集做預測

mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)
adj_r2 = 1 - (1 - r2) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)
pearson = pearsonr(y_test, y_test_pred)[0]
p_value = pearsonr(y_test, y_test_pred)[1]  # 取得 p-value

print(f"測試集結果：")
print(f"MSE:{mse:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"R²: {r2:.3f}")
print(f"adjusted r2:{adj_r2:.3f}")
print(f"Pearson r: {pearson:.3f}")
print(f"Pearson p-value: {p_value:.3f}")

# 計算交叉驗證 R² 分數
cv_scores = cross_val_score(bayes_search.best_estimator_, X_train, y_train, cv=5, scoring='r2')
mean_score = np.mean(cv_scores)
std_score = np.std(cv_scores)

print(f"最佳交叉驗證r2: {mean_score:.4f} ± {std_score:.4f}")

### PCA

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. 使用 PCA 降維訓練資料
pca = PCA(n_components=0.8)  # 保留90%變異量
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)  # 測試資料也要 transform！

print("主成分數量：", pca.n_components_)
print("各個特徵變異量：", pca.explained_variance_ratio_)
print("保留的總變異量：", pca.explained_variance_ratio_.sum())


# 2. 繪製主成分貢獻度圖（可選）
try:
    feature_names = X_train.columns.tolist()
except:
    feature_names = [f"Feature {i}" for i in range(X_train.shape[1])]

num_components_to_plot = pca.n_components_  # 顯示前幾個主成分
top_n_features = 10         # 每個主成分顯示前幾名貢獻特徵

plt.figure(figsize=(15, 5 * num_components_to_plot))

for i in range(num_components_to_plot):
    # 取得該主成分對所有特徵的權重
    component_weights = pca.components_[i]
    
    # 根據貢獻度絕對值排序，取得前 top_n 名的索引
    top_indices = np.argsort(np.abs(component_weights))[::-1][:top_n_features]
    
    # 取得對應特徵名稱與權重
    top_features = [feature_names[j] for j in top_indices]
    top_weights = component_weights[top_indices]

    # 繪製橫條圖
    plt.figure(figsize=(8, 4))
    bars = plt.barh(top_features, top_weights, color='skyblue')
    plt.title(f"Top {top_n_features} Contributing Features to PC{i+1}")
    plt.xlabel("Contribution Weight")
    plt.ylabel("Feature")
    plt.grid(True, axis='x')

    # 加上數值標籤
    for bar in bars:
        width = bar.get_width()
        plt.text(width, bar.get_y() + bar.get_height()/2, f'{width:.2f}', va='center')

    plt.tight_layout()
    plt.show()


### PCA後測試

In [ ]:
ada_model.fit(X_train_pca, y_train)

# --- 預測 ---
y_train_pred = ada_model.predict(X_train_pca)
y_pred = ada_model.predict(X_test_pca)

# --- 訓練集評估 ---
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
mae_train = mean_absolute_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)
adjusted_r2_train = 1 - (1 - r2_train) * (len(y_train) - 1) / (len(y_train) - X_train_pca.shape[1] - 1)

print("\n=== PCA主成分-訓練集評估===")
print("MSE:", mse_train)
print("RMSE:", rmse_train)
print("MAE:", mae_train)
print("R²:", r2_train)
print("Adjusted R²:", adjusted_r2_train)

# --- 測試集評估 ---
mse_test = mean_squared_error(y_test, y_pred)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(y_test, y_pred)
r2_test = r2_score(y_test, y_pred)
adjusted_r2_test = 1 - (1 - r2_test) * (len(y_test) - 1) / (len(y_test) - X_test_pca.shape[1] - 1)

print("\n=== PCA主成分-測試集評估===")
print("MSE:", mse_test)
print("RMSE:", rmse_test)
print("MAE:", mae_test)
print("R²:", r2_test)
print("Adjusted R²:", adjusted_r2_test)

# XGBoost

In [ ]:
"""建立 XGBoost 模型"""
xgb_model = XGBRegressor(
    n_estimators= 300, # 100
    max_depth= 5, # 3
    learning_rate= 0.1, # 0.1
    subsample= 0.8, # 0.8
    colsample_bytree= 0.8, # 0.5
    reg_alpha= 1 ,        # 1 L1 正則化
    reg_lambda= 1 ,       # 1 0.8 L2 正則化
    random_state=42 
)

xgb_model.fit(X_train, y_train)
y_train_pred = xgb_model.predict(X_train) # 預測：訓練集
y_test_pred = xgb_model.predict(X_test) # 預測：測試集

"""訓練集 評估指標"""
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
mae_train = mean_absolute_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)
adj_r2_train = 1 - (1 - r2_train) * (len(y_train) - 1) / (len(y_train) - X_train.shape[1] - 1)
pearson_r_train = pearson_r(y_train, y_train_pred)
pearson_p_train = pearsonr(y_train, y_train_pred)[1]  # 取得 p-value

"""測試集-評估指標"""
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)
adj_r2_test = 1 - (1 - r2_test) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)
pearson_r_test = pearson_r(y_test, y_test_pred)
pearson_p_test = pearsonr(y_test, y_test_pred)[1]  # 取得 p-value

"""顯示結果"""
print("===== XGBoost 訓練集 (Train Set) 評估 =====")

print(f"Train MSE: {mse_train:.3f}")
print(f"Train RMSE: {rmse_train:.3f}")
print(f"Train MAE: {mae_train:.3f}")
print(f"Train R²: {r2_train:.3f}")
print(f"Train Adjusted R²: {adj_r2_train:.3f}")
print(f"Train Pearson r: {pearson_r_train:.3f}")
print(f"Train Pearson p-value: {pearson_p_train:.3f}\n")

print("===== XGBoost 測試集 (Test Set) 評估 =====")
print(f"Test MSE: {mse_test:.3f}")
print(f"Test RMSE: {rmse_test:.3f}")
print(f"Test MAE: {mae_test:.3f}")
print(f"Test R²: {r2_test:.3f}")
print(f"Test Adjusted R²: {adj_r2_test:.3f}")
print(f"Test Pearson r: {pearson_r_test:.3f}")
print(f"Test Pearson p-value: {pearson_p_test:.3f}\n")

# 計算實際值的標準差 
std_dev_train = np.std(y_train)
lower_bound = y_train - 1 * std_dev_train
upper_bound = y_train + 1 * std_dev_train
print("std_dev_train:",std_dev_train)

"""畫圖：訓練集"""
plt.figure(figsize=(4, 4))
plt.scatter(y_train, y_train_pred, color='blue', alpha=0.6, label='Train')
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'k--', label='Ideal')
# 畫出 ±2σ 的容忍區間
x_range = np.linspace(y_train.min(), y_train.max(), 100)
# plt.plot(x_range, x_range + 1 * std_dev_train, 'r--', label='+1σ')
# plt.plot(x_range, x_range - 1 * std_dev_train, 'r--', label='-1σ')
# plt.fill_between(x_range, x_range - 1 * std_dev_train, x_range + 1 * std_dev_train, color='red', alpha=0.1, label='Tolerance ±2σ')

plt.title('XGBoost - Training Set')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.legend()
plt.tight_layout()
plt.show()

# 計算實際值的標準差 (這裡以 y_test 為例)
std_dev_test = np.std(y_test)
lower_bound = y_test - 1 * std_dev_test
upper_bound = y_test + 1 * std_dev_test
print("std_dev_test:", std_dev_test)

"""畫圖：訓練集"""
plt.figure(figsize=(4, 4))
plt.scatter(y_test, y_test_pred, color='green', alpha=0.6, label='Test')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', label='Ideal')

# 畫出 ±2σ 的容忍區間
x_range = np.linspace(y_test.min(), y_test.max(), 100)
# plt.plot(x_range, x_range + 1 * std_dev_test, 'r--', label='+ σ')
# plt.plot(x_range, x_range - 1 * std_dev_test, 'r--', label='- σ')
# plt.fill_between(x_range, x_range - 1 * std_dev_test, x_range + 1 * std_dev_test, color='red', alpha=0.1, label='Tolerance ± σ')

plt.title('XGBoost - Testing Set')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.legend()
plt.tight_layout()
plt.show()


### 五折交叉驗證

In [ ]:
"""五折交叉驗證(輸出每一折的所有評估指標結果)-xgb-"""
cv = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
r2_scores = []
results = []

for train_index, test_index in cv.split(X):
    # 正確用法：使用 iloc 依照列編號取出資料
    X_train_cv = X.iloc[train_index]
    X_test_cv = X.iloc[test_index]
    y_train_cv = y.iloc[train_index]
    y_test_cv = y.iloc[test_index]

    # 模型訓練
    xgb_model.fit(X_train_cv, y_train_cv)

    # 預測
    y_train_pred = xgb_model.predict(X_train_cv)
    y_test_pred = xgb_model.predict(X_test_cv)

    # 訓練集評估
    mse_train = mean_squared_error(y_train_cv, y_train_pred)
    rmse_train = np.sqrt(mse_train)
    mae_train = mean_absolute_error(y_train_cv, y_train_pred)
    r2_train = r2_score(y_train_cv, y_train_pred) 
    adj_r2_train = 1 - (1 - r2_train) * (len(y_train_cv) - 1) / (len(y_train_cv) - X_train_cv.shape[1] - 1)
    pearson_r_train = pearson_r(y_train_cv, y_train_pred)
    p_value_train = pearsonr(y_train_cv, y_train_pred)[1]  # 取得 p-value

    # 測試集評估
    mse_test = mean_squared_error(y_test_cv, y_test_pred)
    rmse_test = np.sqrt(mse_test)
    mae_test = mean_absolute_error(y_test_cv, y_test_pred)
    r2_test = r2_score(y_test_cv, y_test_pred) 
    adj_r2_test = 1 - (1 - r2_test) * (len(y_test_cv) - 1) / (len(y_test_cv) - X_test_cv.shape[1] - 1)
    pearson_r_test = pearson_r(y_test_cv, y_test_pred)
    p_value_test = pearsonr(y_test_cv, y_test_pred)[1]  # 取得 p-value

    # 輸出每折結果
    print(f"===== XGBoost Fold {fold} =====")
    print(f"[Train] \n MSE: {mse_train:.4f} \n RMSE: {rmse_train:.4f} \n MAE: {mae_train:.4f} \n R²: {r2_train:.4f} \n Adj R²: {adj_r2_train:.4f} \n Pearson r: {pearson_r_train:.4f}\n Pearson p-value: {p_value_train:.4f}\n")
    print(f"\n [Test ] \n MSE: {mse_test:.4f} \n RMSE: {rmse_test:.4f} \n MAE: {mae_test:.4f} \n R²: {r2_test:.4f} \n Adj R²: {adj_r2_test:.4f}\n Pearson r: {pearson_r_test:.4f}\n Pearson p-value: {p_value_test:.4f}\n")

    # 可選擇儲存所有指標到列表中以利後續統計分析
    results.append({
        "Fold": fold,
        "Train_MSE": mse_train.round(3),
        "Train_RMSE": rmse_train.round(3),
        "Train_MAE": mae_train.round(3),
        "Train_R2": r2_train.round(3),
        "Train_Adj_R2": adj_r2_train.round(3),
        "Train_Pearson_r": pearson_r_train.round(3),
        "Train_p_value": pearson_p_train.round(3),
        
        "Test_MSE": mse_test.round(2),
        "Test_RMSE": rmse_test.round(2),
        "Test_MAE": mae_test.round(2),
        "Test_R2": r2_test,
        "Test_Adj_R2": adj_r2_test,
        "Test_Pearson_r": pearson_r_test.round(3),
        "Test_p_value": pearson_p_test.round(3)
    })
    
    # 建立左右並排的子圖
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))  # 1列2欄子圖

    # 畫圖：訓練集 True vs Predicted
    axes[0].scatter(y_train_cv, y_train_pred, alpha=0.7, edgecolors='g')
    axes[0].plot([y_train_cv.min(), y_train_cv.max()], [y_train_cv.min(), y_train_cv.max()], 'r--')
    axes[0].set_title(f"Fold {fold} - Train")
    axes[0].set_xlabel("True Values")
    axes[0].set_ylabel("Predicted Values")
    axes[0].grid(True)

    # 畫圖：測試集 True vs Predicted
    axes[1].scatter(y_test_cv, y_test_pred, alpha=0.7, edgecolors='b')
    axes[1].plot([y_test_cv.min(), y_test_cv.max()], [y_test_cv.min(), y_test_cv.max()], 'r--')
    axes[1].set_title(f"Fold {fold} - Test")
    axes[1].set_xlabel("True Values")
    axes[1].set_ylabel("Predicted Values")
    axes[1].grid(True)

    # 整體排版
    plt.suptitle(f"Fold {fold} - True vs. Predicted")
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # 預留上方空間給 suptitle
    plt.show()

    fold += 1

# # 輸出交叉驗證整體結果
# print("===== 5-Fold Cross-Validation Summary =====")
# print(f"平均 R² (測試集): {np.mean(r2_scores):.4f}")
# print(f"標準差 R² (測試集): {np.std(r2_scores):.4f}")

# 將結果轉成 DataFrame
results_df = pd.DataFrame(results)

# 顯示每個指標的平均與標準差（分訓練與測試集）
mean_results = results_df.mean(numeric_only=True)
std_results = results_df.std(numeric_only=True)

print("===== 指標平均與標準差 (5折) =====")
for col in results_df.columns:
    if col != "Fold":
        mean_val = mean_results[col]
        std_val = std_results[col]
        print(f"{col}: 平均 = {mean_val:.4f}，標準差 = {std_val:.4f}")

# 輸出成csv
results_df.to_csv("cv_results_xgb.csv", index=False, encoding='utf-8-sig')
print("\n已儲存每折交叉驗證結果至 'cv_results_xgb.csv'")

# 橫向格式 summary
summary_df = pd.DataFrame({
    metric: [f"{mean:.2f}({std:.2f})"]
    for metric, mean, std in zip(mean_results.index, mean_results, std_results)
})

# 選擇性：加入模型名稱作為 row index
summary_df.insert(0, 'Model', 'AdaBoost')  # ← 若是其他模型名稱也可更改

# 顯示結果
print("\n===== 五折交叉驗證統計（橫向 mean(std) 格式）=====")
print(summary_df)

# 儲存為 CSV
summary_df.to_csv("cv_summary_xgb.csv", index=False, encoding='utf-8-sig')
print("✅ 已儲存橫向 mean(std) 格式至 'cv_summary_xgb.csv'")

### 移動平均後的Pearson相關係數

In [ ]:
"""移動平均後的Pearson r"""

# 6. 將預測結果與識別欄位合併
results = pd.DataFrame({
    'case': identifiers_test['case'],
    'seg': identifiers_test['segment'],
    'true': y_test,
    'pred': y_test_pred,
})

# 7. 按 case 和 seg 排序
results = results.sort_values(by=['case', 'seg']).reset_index(drop=True)

# 自定義移動平均函數
def custom_moving_average(values):
    """
    計算自定義的移動平均值。
    values: 一個數值列表或 numpy array。
    返回: 一個包含移動平均值的列表。
    """
    moving_averages = []
    for i, value in enumerate(values):
        if i == 0:
            # 第一個值直接加入
            moving_averages.append(value)
        else:
            # 後續值基於前一次的平均值計算
            new_average = (moving_averages[-1] + value) / 2
            moving_averages.append(new_average)
    return moving_averages

# 8. 按 case 編號進行移動平均（使用自定義函數）
results['true_avg'] = results.groupby('case')['true'].transform(custom_moving_average)
results['pred_avg'] = results.groupby('case')['pred'].transform(custom_moving_average)

# results['true_avg'] = results.groupby('case')['true'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
# results['pred_avg'] = results.groupby('case')['pred'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

# 9. 保留每個 case 的最後一筆移動平均值（可選）
final_results = results.groupby('case').tail(1).reset_index(drop=True)

# 若需要將結果存回 CSV 檔案
output_csv_name = 'true_pred_xgb.csv'
results.to_csv(output_csv_name, index=False,encoding='utf-8-sig')
print(f"結果已輸出至：{output_csv_name}")

#=====================================================
# 計算 Pearson 相關係數
pearson_r_avg_test, p_value_avg_test = pearsonr(final_results['true_avg'], final_results['pred_avg'])
print(f"Test true_pred_avg Pearson r: {pearson_r_avg_test:.4f} (p={p_value_avg_test:.4f})")

# 繪製散佈圖
plt.figure(figsize=(5, 5))
sns.regplot(x=final_results['true_avg'], 
            y=final_results['pred_avg'], 
            scatter_kws={'alpha': 0.6}, 
            line_kws={'color': 'red'})

# 加入 45 度線
min_val = min(final_results['true_avg'].min(), final_results['pred_avg'].min())
max_val = max(final_results['true_avg'].max(), final_results['pred_avg'].max())
plt.plot([min_val, max_val], [min_val, max_val], color='gray', linestyle='--', linewidth=1.2, label='45° line')

plt.title(f"Average Pearson r = {pearson_r_avg_test:.4f} (p = {p_value_avg_test:.4e})")
plt.xlabel("Average True Values")
plt.ylabel("Average Predicted Values")
plt.grid(True)
plt.tight_layout()
plt.show()

### 輸出csv

In [ ]:
"""輸出csv，參數、模型評估指標、pearson 相關係數"""

model_params = (
    f"n_estimators={xgb_model.n_estimators}, "
    f"max_depth={xgb_model.max_depth}, "
    f"learning_rate={xgb_model.learning_rate}, "
    f"subsample={xgb_model.subsample}, "
    f"colsample_bytree={xgb_model.colsample_bytree}, "
    f"reg_alpha={xgb_model.reg_alpha}, "
    f"reg_lambda={xgb_model.reg_lambda}, "
    f"random_state={xgb_model.random_state}"
)

file_name = os.path.basename(path) # 提取文件名（去掉路径，只保留文件名）

# 建立結果字典
results_dict = {
    "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),  # 加入時間戳記
    "Model": "Random Forest Regressor",
    "path": file_name,
    "y":y.name,
    "Train_MSE": round(mse_train,3),
    "Train_RMSE": round(rmse_train,3),
    "Train_MAE": round(mae_train,3),
    "Train_R2": round(r2_train,3),
    "Train_Adjusted_R2": round(adj_r2_train,3),
    "Train_Pearson_r": round(pearson_r_train, 3),
    "Train_p_value": round(p_value_train, 3),

    "Test_MSE": round(mse_test,3),
    "Test_RMSE": round(rmse_test,3),
    "Test_MAE": round(mae_test,3),
    "Test_R2": round(r2_test,3),
    "Test_Adjusted_R2": round(adj_r2_test,3),
    "Test_Pearson_r": round(pearson_r_test, 3),
    "Train_p_value": round(p_value_test, 3),

    # "Avg_test_Pearson_r": round(pearson_r_avg_test, 3),
    # "Avg_test_Pearson_p": round(p_value_avg_test, 3),
    "model_params": model_params,
}

results_df = pd.DataFrame([results_dict]) # 將結果轉換為 DataFrame
output_csv_name = "train_test_xgb.csv" # 輸出檔案名稱

# 若檔案已存在，則附加資料；否則建立新檔案
if os.path.exists(output_csv_name):
    results_df.to_csv(output_csv_name, mode='a', header=False, index=False)
else:
    results_df.to_csv(output_csv_name, index=False)

print(f"結果已輸出至：{output_csv_name}")

### Shapley Value

In [ ]:
import shap
import matplotlib.pyplot as plt
import pandas as pd

"""Permutation SHAP"""
# # 回歸模型預測函數：直接使用 predict() 回傳連續數值
# xgb_predict_fn = lambda x: xgb_model.predict(pd.DataFrame(x, columns=X_test.columns))

# # 建立 explainer 並取得 SHAP 值（回歸問題）
# explainer_xgb = shap.PermutationExplainer(xgb_predict_fn, X_test)
# shap_values_xgb = explainer_xgb(X_test)

"""Tree SHAP"""
# 回歸模型預測函數：直接使用 predict() 回傳連續數值
# xgb_predict_fn = lambda x: xgb_model.predict(pd.DataFrame(x, columns=X_test.columns))
explainer_xgb = shap.Explainer(xgb_model, X_test) 
shap_values_xgb = explainer_xgb(X_test)

top_n = 15
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_rf.values, X_test, show=False, max_display= top_n)

# plt.title("Random Forest Feature Importance (Permutation SHAP)")
plt.title("XGBoost (Tree SHAP)")
plt.show()

### 特徵重要性分析

In [ ]:
""""特徵重要性分析"""
importances = xgb_model.feature_importances_  # 取得特徵重要性
feature_names = X_train.columns              # 特徵名稱
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# 顯示 Top 10
print("\n===== 特徵重要性 (Top 20) =====")
print(importance_df.head(20))

# 畫圖：Top 10
plt.figure(figsize=(10, 10))
plt.barh(importance_df['Feature'][:20][::-1], importance_df['Importance'][:20][::-1], color='steelblue')
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances ()')
plt.tight_layout()
plt.show()

### 貝氏優化

In [ ]:
"""XGBoost 貝氏優化"""
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from skopt import BayesSearchCV
from skopt.space import Integer, Real
import numpy as np

# 建立 XGBoost 回歸模型
# model = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1)

# 定義搜尋空間（針對小資料集設計）
param_space = {
    'n_estimators': Integer(100, 500),                # 弱學習器數量（不要太大）
    'learning_rate': Real(0.01, 0.1, prior='log-uniform'),  # 學習率，小資料建議偏小
    'max_depth': Integer(2, 8),                      # 控制樹深度，小資料不宜太深
    # 'subsample': Real(0.6, 1.0),                     # 訓練子樣本比例（避免過擬合）
    # 'colsample_bytree': Real(0.6, 1.0),              # 特徵子集比例（避免過擬合）
    # 'min_child_weight': Integer(1, 10),              # 葉節點最小樣本權重（越大越保守）
    # 'gamma': Real(0, 1.0),                           # 損失下降要求，越大越保守
    # 'reg_alpha': Real(0.0, 1.0),                     # L1 正則化（特徵選擇）
    # 'reg_lambda': Real(0.0, 1.0)                     # L2 正則化（防止過擬合）
}

# 執行 BayesSearchCV
bayes_search = BayesSearchCV(
    estimator= xgb_model,
    search_spaces=param_space,
    n_iter=100,             # 搜尋次數（可根據時間與需求調整）
    cv=5,                   # 五折交叉驗證
    scoring= 'r2',           # 回歸評估指標
    random_state=42,
    # n_jobs=-1,
)

# 用訓練資料幫你自動找出「最佳參數組合」
bayes_search.fit(X_train, y_train)

# 取得最佳參數和分數
print("最佳參數:")
print(bayes_search.best_params_)

# 直接使用最佳模型做預測（推薦）
best_model = bayes_search.best_estimator_ # 取出已訓練好的最佳模型

# 在測試集上預測
y_test_pred = best_model.predict(X_test) # 用測試集做預測

mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)
adj_r2 = 1 - (1 - r2) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)
pearson = pearsonr(y_test, y_test_pred)[0]
p_value = pearsonr(y_test, y_test_pred)[1]  # 取得 p-value

print(f"測試集結果：")
print(f"MSE:{mse:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"R²: {r2:.3f}")
print(f"adjusted r2:{adj_r2:.3f}")
print(f"Pearson r: {pearson:.3f}")
print(f"Pearson p-value: {p_value:.3f}")

# 計算交叉驗證 R² 分數
cv_scores = cross_val_score(bayes_search.best_estimator_, X_train, y_train, cv=5, scoring='r2')
mean_score = np.mean(cv_scores)
std_score = np.std(cv_scores)

print(f"最佳交叉驗證r2: {mean_score:.4f} ± {std_score:.4f}")

# 建立結果字典
results_dict = {
    "scoring": bayes_search.scoring,
    "Best_Params": bayes_search.best_params_,
    "Mean_CV_Score": np.mean(cv_scores),
    "Std_CV_Score": np.std(cv_scores),
}

# 將結果轉換為 DataFrame
result_df = pd.DataFrame([results_dict])

# 輸出檔案名稱（你可以改成絕對路徑或手動指定路徑）
output_csv_name = "bayes_xgb.csv"

# 若已有檔案就 append，否則建立新檔
if os.path.exists(output_csv_name):
    result_df.to_csv(output_csv_name, mode='a', header=False, index=False)
else:
    result_df.to_csv(output_csv_name, index=False)

print(f"✅ 貝氏優化結果已輸出至：{output_csv_name}")

### PCA

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. 使用 PCA 降維訓練資料
pca = PCA(n_components=0.9)  # 保留90%變異量
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)  # 測試資料也要 transform！

print("主成分數量：", pca.n_components_)
print("各個特徵變異量：", pca.explained_variance_ratio_)
print("保留的總變異量：", pca.explained_variance_ratio_.sum())


# 2. 繪製主成分貢獻度圖（可選）
try:
    feature_names = X_train.columns.tolist()
except:
    feature_names = [f"Feature {i}" for i in range(X_train.shape[1])]

num_components_to_plot = pca.n_components_  # 顯示前幾個主成分
top_n_features = 10         # 每個主成分顯示前幾名貢獻特徵

plt.figure(figsize=(15, 5 * num_components_to_plot))

for i in range(num_components_to_plot):
    # 取得該主成分對所有特徵的權重
    component_weights = pca.components_[i]
    
    # 根據貢獻度絕對值排序，取得前 top_n 名的索引
    top_indices = np.argsort(np.abs(component_weights))[::-1][:top_n_features]
    
    # 取得對應特徵名稱與權重
    top_features = [feature_names[j] for j in top_indices]
    top_weights = component_weights[top_indices]

    # 繪製橫條圖
    plt.figure(figsize=(8, 4))
    bars = plt.barh(top_features, top_weights, color='skyblue')
    plt.title(f"Top {top_n_features} Contributing Features to PC{i+1}")
    plt.xlabel("Contribution Weight")
    plt.ylabel("Feature")
    plt.grid(True, axis='x')

    # 加上數值標籤
    for bar in bars:
        width = bar.get_width()
        plt.text(width, bar.get_y() + bar.get_height()/2, f'{width:.2f}', va='center')

    plt.tight_layout()
    plt.show()


### PCA後測試

In [ ]:
xgb_model.fit(X_train_pca, y_train)

# --- 預測 ---
y_train_pred = xgb_model.predict(X_train_pca)
y_pred = xgb_model.predict(X_test_pca)

# --- 訓練集評估 ---
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
mae_train = mean_absolute_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)
adjusted_r2_train = 1 - (1 - r2_train) * (len(y_train) - 1) / (len(y_train) - X_train_pca.shape[1] - 1)

print("\n=== PCA主成分-訓練集評估===")
print("MSE:", mse_train)
print("RMSE:", rmse_train)
print("MAE:", mae_train)
print("R²:", r2_train)
print("Adjusted R²:", adjusted_r2_train)

# --- 測試集評估 ---
mse_test = mean_squared_error(y_test, y_pred)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(y_test, y_pred)
r2_test = r2_score(y_test, y_pred)
adjusted_r2_test = 1 - (1 - r2_test) * (len(y_test) - 1) / (len(y_test) - X_test_pca.shape[1] - 1)

print("\n=== PCA主成分-測試集評估===")
print("MSE:", mse_test)
print("RMSE:", rmse_test)
print("MAE:", mae_test)
print("R²:", r2_test)
print("Adjusted R²:", adjusted_r2_test)

# Shapley value

In [ ]:
# pip install shap
# pip install numpy

In [ ]:
import shap
import matplotlib.pyplot as plt
import pandas as pd

# 回歸模型預測函數：直接使用 predict() 回傳連續數值
ada_predict_fn = lambda x: ada_model.predict(pd.DataFrame(x, columns=X_test.columns))
rf_predict_fn = lambda x: rf_model.predict(pd.DataFrame(x, columns=X_test.columns))
xgb_predict_fn = lambda x: xgb_model.predict(pd.DataFrame(x, columns=X_test.columns))
# lgbm_predict_fn = lambda x: lgbm_model.predict(pd.DataFrame(x, columns=X_test.columns))

# 建立 explainer 並取得 SHAP 值（回歸問題）
explainer_rf = shap.PermutationExplainer(rf_predict_fn, X_test)
# explainer_rf = shap.Explainer(rf_model, X_test) 
shap_values_rf = explainer_rf(X_test)

# Random Forest
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_rf.values, X_test, show=False)
plt.title("Random Forest Feature Importance (PermutationExplainer - Regression)")
plt.show()



In [ ]:
# AdaBoost

explainer_ada = shap.PermutationExplainer(ada_predict_fn, X_test)
shap_values_ada = explainer_ada(X_test)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_ada.values, X_test, show=False)
plt.title("AdaBoost Feature Importance (PermutationExplainer - Regression)")
plt.show()


In [ ]:
# XGBoost

explainer_xgb = shap.PermutationExplainer(xgb_predict_fn, X_test)
shap_values_xgb = explainer_xgb(X_test)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_xgb.values, X_test, show=False)
plt.title("XGBoost Feature Importance (PermutationExplainer - Regression)")
plt.show()

# RFECV (Recursive Feature Elimination with Cross-Validation)

In [ ]:
## RFECV特徵選擇
from sklearn.feature_selection import RFECV

# model = XGBRegressor()

# 建立 RFECV 物件
rfecv = RFECV(estimator = xgb_model,
        min_features_to_select = 30,
        step = 5,
        cv = 5,
        scoring = pearson_scorer,
        # scoring = 'r2',
        # verbose = 1
        )

# 執行 RFECV
rfecv.fit(X_train, y_train)

# 特徵挑選過程
rfecv.cv_results_

#畫出 RFECV 每一次特徵數下的模型效果圖
# 如果你是用 neg_MSE 或 r2，可以直接畫分數
plt.figure(figsize=(10, 5))
plt.title("RFECV - Model Score vs Number of Features")
plt.xlabel("Number of Selected Features")
plt.ylabel("Cross-validation Score")

# 根據你用的是負 MSE 還是 Pearson，自行決定是否取負號
# scores = rfecv.cv_results_[pearson_scorer].mean()  # 取平均分數
# scores = rfecv.cv_results_["mean_test_score"]
# scores = -scores if pearson_scorer == 'neg_mean_squared_error' else scores
scores = rfecv.cv_results_["mean_test_score"]
num_features = range(rfecv.min_features_to_select, 
                     rfecv.min_features_to_select + len(scores) * rfecv.step, 
                     rfecv.step)
plt.plot(num_features, scores, marker='o')
plt.axvline(rfecv.n_features_, linestyle='--', color='red', label=f"Best: {rfecv.n_features_} features")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 取得交叉驗證所有折的分數欄位
fold_scores = [col for col in rfecv.cv_results_ if "split" in col and "test_score" in col]

# 建立 DataFrame 顯示
cv_score_df = pd.DataFrame({
    "n_features": [X_train.shape[1] - i*rfecv.step for i in range(len(rfecv.cv_results_["mean_test_score"]))],
})

# 加入每折分數
for fold in fold_scores:
    cv_score_df[fold] = rfecv.cv_results_[fold]

# 顯示前幾列結果
print("\nRFECV 每個特徵數下的每一折分數：")
print(cv_score_df.head())

selected_mask = rfecv.support_  # 一個 boolean array，表示哪個特徵被保留
selected_features = X_train.columns[selected_mask]
print("保留的特徵數：", len(selected_features))
print("保留的特徵名稱：\n", selected_features.tolist())

removed_features = X_train.columns[~rfecv.support_]
print("\n被排除的特徵數：", len(removed_features))
print("被排除的特徵名稱：\n", removed_features.tolist())

# # 被挑選出來的特徵位置
# selected_features = rfecv.get_feature_names_out()
# print("被挑選出來的特徵名稱：", selected_features)

# 模型在選擇後特徵上的效果
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

rf_model.fit(X_train_sel, y_train)
y_pred_sel = rf_model.predict(X_test_sel)

final_r = pearson_r(y_test, y_pred_sel)
print("選擇特徵後的 Pearson r：", round(final_r, 4))


# 主成分分析降維 PCA

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. 使用 PCA 降維訓練資料
pca = PCA(n_components=0.8)  # 保留90%變異量
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)  # 測試資料也要 transform！

print("主成分數量：", pca.n_components_)
print("各個特徵變異量：", pca.explained_variance_ratio_)
print("保留的總變異量：", pca.explained_variance_ratio_.sum())


# 2. 繪製主成分貢獻度圖（可選）
try:
    feature_names = X_train.columns.tolist()
except:
    feature_names = [f"Feature {i}" for i in range(X_train.shape[1])]

num_components_to_plot = pca.n_components_  # 顯示前幾個主成分
top_n_features = 10         # 每個主成分顯示前幾名貢獻特徵

plt.figure(figsize=(15, 5 * num_components_to_plot))

for i in range(num_components_to_plot):
    # 取得該主成分對所有特徵的權重
    component_weights = pca.components_[i]
    
    # 根據貢獻度絕對值排序，取得前 top_n 名的索引
    top_indices = np.argsort(np.abs(component_weights))[::-1][:top_n_features]
    
    # 取得對應特徵名稱與權重
    top_features = [feature_names[j] for j in top_indices]
    top_weights = component_weights[top_indices]

    # 繪製橫條圖
    plt.figure(figsize=(8, 4))
    bars = plt.barh(top_features, top_weights, color='skyblue')
    plt.title(f"Top {top_n_features} Contributing Features to PC{i+1}")
    plt.xlabel("Contribution Weight")
    plt.ylabel("Feature")
    plt.grid(True, axis='x')

    # 加上數值標籤
    for bar in bars:
        width = bar.get_width()
        plt.text(width, bar.get_y() + bar.get_height()/2, f'{width:.2f}', va='center')

    plt.tight_layout()
    plt.show()


# 測試主成份效果

In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import matplotlib.pyplot as plt

# # --- PCA 降維 ---
# pca = PCA(n_components=0.9)
# X_train_pca = pca.fit_transform(X_train)
# X_test_pca = pca.transform(X_test)

# --- 建立並訓練隨機森林模型 ---
# rf_model = RandomForestRegressor(random_state=42)
xgb_model.fit(X_train_pca, y_train)

# --- 預測 ---
y_train_pred = xgb_model.predict(X_train_pca)
y_pred = xgb_model.predict(X_test_pca)

# --- 訓練集評估 ---
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
mae_train = mean_absolute_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)
adjusted_r2_train = 1 - (1 - r2_train) * (len(y_train) - 1) / (len(y_train) - X_train_pca.shape[1] - 1)

print("\n=== PCA主成分-訓練集評估===")
print("MSE:", mse_train)
print("RMSE:", rmse_train)
print("MAE:", mae_train)
print("R²:", r2_train)
print("Adjusted R²:", adjusted_r2_train)

# --- 測試集評估 ---
mse_test = mean_squared_error(y_test, y_pred)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(y_test, y_pred)
r2_test = r2_score(y_test, y_pred)
adjusted_r2_test = 1 - (1 - r2_test) * (len(y_test) - 1) / (len(y_test) - X_test_pca.shape[1] - 1)

print("\n=== PCA主成分-測試集評估===")
print("MSE:", mse_test)
print("RMSE:", rmse_test)
print("MAE:", mae_test)
print("R²:", r2_test)
print("Adjusted R²:", adjusted_r2_test)

In [ ]:
"""Pearson 相關係數(預測vs真實)(改！！！！)"""
from scipy.stats import pearsonr
import seaborn as sns

pearson_r_train, p_value_train = pearsonr(y_train, y_train_pred)
print(f"Train Pearson r: {pearson_r_train:.4f} (p={p_value_train:.4f})")

pearson_r_test, p_value_test = pearsonr(y_test, y_test_pred)
print(f"Test Pearson r: {pearson_r_test:.4f} (p={p_value_test:.4f})")

# ===== Pearson 相關性圖（訓練集）=====
plt.figure(figsize=(4, 4))
sns.regplot(x=y_train, y=y_train_pred, scatter_kws={'alpha':0.6}, line_kws={'color':'red'})
r_train, p_train = pearsonr(y_train, y_train_pred)

# 加入 45 度線
min_val = min(y_train.min(), y_train_pred.min())
max_val = max(y_train.max(), y_train_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], color='gray', linestyle='--', linewidth=1.2, label='45° line')

plt.title(f"Train Set: Pearson r = {pearson_r_train:.4f} (p = {p_value_train:.4e})")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.grid(True)
plt.tight_layout()
plt.show()

# ===== Pearson 相關性圖（測試集）=====
plt.figure(figsize=(4, 4))
sns.regplot(x=y_test, y=y_test_pred, scatter_kws={'alpha':0.6}, line_kws={'color':'red'})
r_test, p_test = pearsonr(y_test, y_test_pred)

# 加入 45 度線
min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], color='gray', linestyle='--', linewidth=1.2, label='45° line')

plt.title(f"Test Set: Pearson r = {r_test:.4f} (p = {p_test:.4e})")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.grid(True)
plt.tight_layout()
plt.show()